In [ ]:
import os
import requests
import gradio as gr

from dotenv import load_dotenv


# Load the project .env file
ENV_PATH = r"C:\Users\ELEAZAR GIDEON\Documents\jekacode-ai-engineering\.env"

load_dotenv(ENV_PATH, override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODELS_URL = "https://openrouter.ai/api/v1/models"

print("OpenRouter key loaded:", bool(OPENROUTER_API_KEY))


In [ ]:
def list_free_models():

    try:
        response = requests.get(
            MODELS_URL,
            timeout=30
        )

        response.raise_for_status()

        all_models = response.json()["data"]

        free_ids = [
            model["id"]
            for model in all_models
            if model["id"].endswith(":free")
        ]

        free_ids.sort()

    except Exception as error:
        print("Could not fetch free models:", error)
        free_ids = []

    # Always keep OpenRouter's free router as a fallback
    return ["openrouter/free"] + free_ids

In [ ]:
free_models = list_free_models()

print("Free models found:", len(free_models))
print("\nAvailable models:")

for model in free_models[:20]:
    print(model)

In [ ]:
def ask_openrouter(messages, model):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "AI Chatbot",
    }
    payload = {"model": model, "messages": messages}
    r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

In [ ]:
def ask_openrouter(messages, model):

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "AI Chatbot",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": messages
    }

    response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload,
        timeout=90
    )

    if not response.ok:
        print("OpenRouter error:", response.status_code)
        print(response.text[:1000])

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"].strip()

In [ ]:
def chat_fn(message, history, model):

    messages = []

    # Add previous conversation messages
    for turn in history:

        # Newer Gradio history format
        if isinstance(turn, dict):

            role = turn.get("role")
            content = turn.get("content")

            if role in ["user", "assistant"] and content:
                messages.append({
                    "role": role,
                    "content": content
                })

        # Older Gradio history format
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:

            user_message, assistant_message = turn

            if user_message:
                messages.append({
                    "role": "user",
                    "content": user_message
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": assistant_message
                })

    # Add the new message
    messages.append({
        "role": "user",
        "content": message
    })

    try:
        return ask_openrouter(messages, model)

    except requests.exceptions.HTTPError as error:

        if error.response is not None:

            if error.response.status_code == 429:
                return (
                    "Rate limited. Please wait a moment or choose another model."
                )

            if error.response.status_code == 404:
                return (
                    f"Model '{model}' is unavailable. Please choose another model."
                )

            return (
                f"OpenRouter error {error.response.status_code}: "
                f"{error.response.text[:500]}"
            )

        return f"HTTP error: {error}"

    except Exception as error:
        return f"Something went wrong: {error}"

In [ ]:
free_models = list_free_models()

default_model = (
    free_models[0]
    if free_models
    else "openrouter/free"
)

model_picker = gr.Dropdown(
    choices=free_models,
    value=default_model,
    label="Model (free tier)"
)

demo = gr.ChatInterface(
    fn=chat_fn,
    additional_inputs=[model_picker],
    title="AI Chatbot — Free Models via OpenRouter",
    description=(
        "Choose a model from the dropdown and start chatting."
    )
)

if __name__ == "__main__":
    demo.launch()